# So sánh CSV file path với folder theo IB + version + file type

Notebook này chỉ có **1 code cell**. Sửa phần `CONFIG` ở đầu cell rồi chạy.

Rule so sánh:
- Extract `IB` dạng `IB` + 10 số.
- Extract `version` dạng `v00-00`, `v01-00`, ...; nếu CSV có cột `version` thì ưu tiên cột đó.
- So sánh theo khóa: `IB + version + file_type`.
- Excel chỉ so với Excel, PDF chỉ so với PDF.
- Nếu có cùng key nhưng tên file khác → `name_mismatch`.
- Nếu thiếu IB/version/type tương ứng trong folder → mismatch tương ứng.

In [21]:
from pathlib import Path, PureWindowsPath
import pandas as pd
import re

# =========================
# CONFIG - SỬA Ở ĐÂY
# =========================
CSV_FILE = Path(r"C:\Users\ncdhuy\Downloads\test\x.csv")
CSV_PATH_COLUMN = "file_path"

TARGET_FOLDER = Path(r"C:\Users\ncdhuy\Downloads\test\raw_data\20260509\lite")
RECURSIVE_SCAN = False

OUTPUT_FILE = Path(r"C:\Users\ncdhuy\Downloads\test\reports\file_path_vs_folder_mismatch_report.xlsx")

EXCEL_EXTENSIONS = {".xlsx", ".xls", ".xlsm", ".xlsb"}
PDF_EXTENSIONS = {".pdf"}

IB_PATTERN = re.compile(r"(IB\d{10})", re.IGNORECASE)


# =========================
# FUNCTIONS
# =========================
def extract_ib(text):
    if pd.isna(text):
        return None
    match = IB_PATTERN.search(str(text))
    return match.group(1).upper() if match else None


def normalize_version(version):
    if pd.isna(version):
        return None

    v = str(version).strip().upper()

    if not v or v in {"NAN", "NONE", "<NA>"}:
        return None

    # Nếu pandas đọc 00 thành 0 hoặc 01 thành 1
    if re.fullmatch(r"\d+", v):
        return "V" + v.zfill(2)

    # Nếu là 0.0, 1.0 do pandas đọc float
    if re.fullmatch(r"\d+\.0", v):
        return "V" + str(int(float(v))).zfill(2)

    # V0 -> V00, V1 -> V01
    match = re.fullmatch(r"V(\d+)", v)
    if match:
        return "V" + match.group(1).zfill(2)

    # V00-00 giữ nguyên, không rút gọn
    v = v.replace(".", "-")

    if re.fullmatch(r"\d{1,3}(?:-\d{1,3})?", v):
        parts = v.split("-")
        parts = [p.zfill(2) for p in parts]
        return "V" + "-".join(parts)

    match = re.fullmatch(r"V(\d{1,3})-(\d{1,3})", v)
    if match:
        return "V" + match.group(1).zfill(2) + "-" + match.group(2).zfill(2)

    return v


def extract_version(text):
    if pd.isna(text):
        return None

    text = str(text)

    # IB2600099365_v00_397_... -> V00
    # IB2500045886_v00-00_...  -> V00-00
    match = re.search(
        r"IB\d{10}[_\-\s]+(v\d{1,3}(?:-\d{1,3})?)",
        text,
        flags=re.IGNORECASE
    )
    if match:
        return normalize_version(match.group(1))

    return None


def get_file_type_from_suffix(suffix):
    suffix = str(suffix).lower()
    if suffix in EXCEL_EXTENSIONS:
        return "excel"
    if suffix in PDF_EXTENSIONS:
        return "pdf"
    return None


def get_file_name_from_any_path(path_value):
    if pd.isna(path_value):
        return None
    text = str(path_value).strip().strip('"')
    if not text:
        return None
    return PureWindowsPath(text).name


def get_suffix_from_name(file_name):
    if not file_name:
        return None
    return Path(str(file_name)).suffix.lower()


def normalize_file_name(name):
    if pd.isna(name):
        return None
    return re.sub(r"\s+", " ", str(name).strip()).lower()


def choose_path_column(df, explicit_col=None):
    if explicit_col:
        if explicit_col not in df.columns:
            raise ValueError(f"Không tìm thấy cột {explicit_col}. Các cột hiện có: {list(df.columns)}")
        return explicit_col

    candidates = []
    for col in df.columns:
        s = df[col].astype(str)
        score = (
            s.str.contains(r"IB\d{10}", case=False, regex=True, na=False).sum()
            + s.str.contains(r"\.(xlsx|xls|xlsm|xlsb|pdf)$", case=False, regex=True, na=False).sum()
            + s.str.contains(r"[A-Za-z]:\\|/", regex=True, na=False).sum()
        )
        candidates.append((score, col))

    candidates.sort(reverse=True)

    if not candidates or candidates[0][0] == 0:
        raise ValueError("Không tự đoán được cột chứa file path. Hãy set CSV_PATH_COLUMN thủ công.")

    return candidates[0][1]


def scan_folder(folder: Path, recursive=True):
    records = []

    if not folder.exists():
        raise FileNotFoundError(f"Không tìm thấy TARGET_FOLDER: {folder}")

    iterator = folder.rglob("*") if recursive else folder.iterdir()

    for file in iterator:
        if not file.is_file():
            continue

        file_type = get_file_type_from_suffix(file.suffix)
        if file_type is None:
            continue

        records.append({
            "folder_file_path": str(file),
            "folder_file_name": file.name,
            "folder_suffix": file.suffix.lower(),
            "folder_file_type": file_type,
            "ib": extract_ib(file.name),
            "version": extract_version(file.name),
            "folder_file_name_norm": normalize_file_name(file.name),
        })

    cols = [
        "folder_file_path",
        "folder_file_name",
        "folder_suffix",
        "folder_file_type",
        "ib",
        "version",
        "folder_file_name_norm",
    ]

    return pd.DataFrame(records, columns=cols)


def join_unique(values):
    vals = [str(v) for v in values if pd.notna(v)]
    return " | ".join(sorted(set(vals))) if vals else None


def to_set(value):
    if pd.isna(value) or value is None or value == "":
        return set()
    return set(str(value).split(" | "))


def diff_set(left, right):
    result = sorted(to_set(left) - to_set(right))
    return " | ".join(result) if result else None


def intersect_set(left, right):
    result = sorted(to_set(left) & to_set(right))
    return " | ".join(result) if result else None


# =========================
# LOAD CSV + PARSE FILE PATH
# =========================
df_csv_raw = pd.read_csv(CSV_FILE, dtype={"version": "string"})
path_col = choose_path_column(df_csv_raw, CSV_PATH_COLUMN)

df_csv = df_csv_raw.copy()
df_csv["csv_row_number"] = range(2, len(df_csv) + 2)
df_csv["csv_file_path"] = df_csv[path_col]
df_csv["csv_file_name"] = df_csv["csv_file_path"].apply(get_file_name_from_any_path)
df_csv["csv_suffix"] = df_csv["csv_file_name"].apply(get_suffix_from_name)
df_csv["csv_file_type"] = df_csv["csv_suffix"].apply(get_file_type_from_suffix)
df_csv["ib"] = df_csv["csv_file_name"].apply(extract_ib)
df_csv["version_from_name"] = df_csv["csv_file_name"].apply(extract_version)

if "version" in df_csv.columns:
    df_csv["version"] = df_csv["version"].apply(normalize_version)
    df_csv["version"] = df_csv["version"].fillna(df_csv["version_from_name"])
else:
    df_csv["version"] = df_csv["version_from_name"]

df_csv["version"] = df_csv["version"].apply(normalize_version)
df_csv["csv_file_name_norm"] = df_csv["csv_file_name"].apply(normalize_file_name)

df_folder = scan_folder(TARGET_FOLDER, recursive=RECURSIVE_SCAN)


# =========================
# DEBUG PREVIEW
# =========================
# print("Sample CSV parsed:")
# display(df_csv[["csv_file_name", "ib", "version", "csv_file_type"]].head(30))

# print("Sample folder parsed:")
# display(df_folder[["folder_file_name", "ib", "version", "folder_file_type"]].head(30))


# =========================
# VALID / INVALID SPLITS
# =========================
df_csv_valid = df_csv.dropna(subset=["ib", "version", "csv_file_type"]).copy()
df_folder_valid = df_folder.dropna(subset=["ib", "version", "folder_file_type"]).copy()

df_csv_no_ib = df_csv[df_csv["ib"].isna()].copy()
df_csv_no_version = df_csv[df_csv["ib"].notna() & df_csv["version"].isna()].copy()
df_csv_unsupported_type = df_csv[
    df_csv["ib"].notna()
    & df_csv["version"].notna()
    & df_csv["csv_file_type"].isna()
].copy()

df_folder_no_ib = df_folder[df_folder["ib"].isna()].copy()
df_folder_no_version = df_folder[
    df_folder["ib"].notna()
    & df_folder["version"].isna()
].copy()


# =========================
# GROUP CSV AND FOLDER BY IB + TYPE
# =========================
csv_group = (
    df_csv_valid
    .groupby(["ib", "csv_file_type"], dropna=False)
    .agg(
        csv_versions=("version", join_unique),
        csv_file_names=("csv_file_name", join_unique),
        csv_file_paths=("csv_file_path", join_unique),
        csv_count=("csv_file_name", "count"),
    )
    .reset_index()
)

folder_group = (
    df_folder_valid
    .groupby(["ib", "folder_file_type"], dropna=False)
    .agg(
        folder_versions=("version", join_unique),
        folder_file_names=("folder_file_name", join_unique),
        folder_file_paths=("folder_file_path", join_unique),
        folder_count=("folder_file_name", "count"),
    )
    .reset_index()
)

df_result = csv_group.merge(
    folder_group,
    left_on=["ib", "csv_file_type"],
    right_on=["ib", "folder_file_type"],
    how="left"
)

df_result["csv_versions_only"] = df_result.apply(
    lambda r: diff_set(r["csv_versions"], r["folder_versions"]),
    axis=1
)

df_result["folder_versions_only"] = df_result.apply(
    lambda r: diff_set(r["folder_versions"], r["csv_versions"]),
    axis=1
)

df_result["versions_intersection"] = df_result.apply(
    lambda r: intersect_set(r["csv_versions"], r["folder_versions"]),
    axis=1
)


def classify_group(row):
    csv_versions = to_set(row["csv_versions"])
    folder_versions = to_set(row["folder_versions"])

    if pd.isna(row["folder_file_names"]):
        return "missing_ib_type_in_folder"

    if csv_versions == folder_versions:
        return "version_set_match"

    if csv_versions & folder_versions:
        return "version_set_partial_mismatch"

    return "version_set_mismatch"


df_result["final_case"] = df_result.apply(classify_group, axis=1)


# =========================
# OPTIONAL: EXACT FILE NAME CHECK BY IB + VERSION + TYPE
# =========================
df_exact_compare = df_csv_valid.merge(
    df_folder_valid,
    left_on=["ib", "version", "csv_file_type"],
    right_on=["ib", "version", "folder_file_type"],
    how="left",
    suffixes=("_csv", "_folder")
)

df_exact_compare["exact_name_match"] = (
    df_exact_compare["folder_file_name"].notna()
    & (df_exact_compare["csv_file_name_norm"] == df_exact_compare["folder_file_name_norm"])
)

df_name_mismatch = df_exact_compare[
    df_exact_compare["folder_file_name"].notna()
    & ~df_exact_compare["exact_name_match"]
].copy()

df_name_mismatch["case"] = "name_mismatch_same_ib_version_type"


# =========================
# EXTRA CHECKS
# =========================
csv_ib_type = set(zip(df_csv_valid["ib"], df_csv_valid["csv_file_type"]))

df_folder_extra_by_ib_type = df_folder_valid[
    ~df_folder_valid.apply(
        lambda r: (r["ib"], r["folder_file_type"]) in csv_ib_type,
        axis=1
    )
].copy()

df_folder_duplicates = df_folder_valid[
    df_folder_valid.duplicated(
        subset=["ib", "version", "folder_file_type"],
        keep=False
    )
].sort_values(["ib", "version", "folder_file_type", "folder_file_name"]).copy()

summary = (
    df_result["final_case"]
    .value_counts(dropna=False)
    .rename_axis("case")
    .reset_index(name="count")
)

mismatch_case_values = {
    "missing_ib_type_in_folder",
    "version_set_partial_mismatch",
    "version_set_mismatch",
}

df_mismatch = df_result[df_result["final_case"].isin(mismatch_case_values)].copy()


# =========================
# EXPORT
# =========================
with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    summary.to_excel(writer, sheet_name="summary", index=False)
    df_mismatch.to_excel(writer, sheet_name="mismatch_versions", index=False)
    df_result.to_excel(writer, sheet_name="all_version_compare", index=False)

    df_name_mismatch.to_excel(writer, sheet_name="name_mismatch", index=False)

    df_csv_no_ib.to_excel(writer, sheet_name="csv_no_ib", index=False)
    df_csv_no_version.to_excel(writer, sheet_name="csv_no_version", index=False)
    df_csv_unsupported_type.to_excel(writer, sheet_name="csv_unsupported_type", index=False)

    df_folder_no_ib.to_excel(writer, sheet_name="folder_no_ib", index=False)
    df_folder_no_version.to_excel(writer, sheet_name="folder_no_version", index=False)
    df_folder_extra_by_ib_type.to_excel(writer, sheet_name="folder_extra_by_ib_type", index=False)
    df_folder_duplicates.to_excel(writer, sheet_name="folder_duplicates", index=False)

    df_csv.to_excel(writer, sheet_name="csv_all_parsed", index=False)
    df_folder.to_excel(writer, sheet_name="folder_all_scanned", index=False)

print("Hoàn tất.")
print(f"Cột file path dùng để so sánh: {path_col}")
print(f"Tổng dòng CSV: {len(df_csv):,}")
print(f"Tổng dòng CSV hợp lệ IB + version + type: {len(df_csv_valid):,}")
print(f"Tổng file Excel/PDF trong folder: {len(df_folder):,}")
print(f"Tổng file folder hợp lệ IB + version + type: {len(df_folder_valid):,}")
print(f"Tổng mismatch version/cần kiểm tra: {len(df_mismatch):,}")
print(f"Tổng name mismatch cùng IB + version + type: {len(df_name_mismatch):,}")
print(f"Đã xuất report: {OUTPUT_FILE.resolve()}")

# display(summary)
# display(df_mismatch.head(50))

Hoàn tất.
Cột file path dùng để so sánh: file_path
Tổng dòng CSV: 400
Tổng dòng CSV hợp lệ IB + version + type: 400
Tổng file Excel/PDF trong folder: 398
Tổng file folder hợp lệ IB + version + type: 398
Tổng mismatch version/cần kiểm tra: 2
Tổng name mismatch cùng IB + version + type: 0
Đã xuất report: C:\Users\ncdhuy\Downloads\test\reports\file_path_vs_folder_mismatch_report.xlsx


In [15]:
tests = [
    "IB2600102293_v00_86_QĐ-BVĐKTL_Danh_sach_hang_hoa_IB2600102293.xlsx",
    "IB2600102293_v01_86_QĐ-BVĐKTL_Danh_sach_hang_hoa_IB2600102293.xlsx",
    "IB2600102293_v02_86_QĐ-BVĐKTL_Danh_sach_hang_hoa_IB2600102293.xlsx",
]

for t in tests:
    print(t, "=>", extract_ib(t), extract_version(t))

IB2600102293_v00_86_QĐ-BVĐKTL_Danh_sach_hang_hoa_IB2600102293.xlsx => IB2600102293 V00
IB2600102293_v01_86_QĐ-BVĐKTL_Danh_sach_hang_hoa_IB2600102293.xlsx => IB2600102293 V01
IB2600102293_v02_86_QĐ-BVĐKTL_Danh_sach_hang_hoa_IB2600102293.xlsx => IB2600102293 V02
